# Re-evaluate `cda_reg` — no retraining

Two things changed since your run, both **evaluation-only**. The training loop
never touches either, so your checkpoint is fine as-is. This rescores it.

**~15 minutes**, not 2 hours.

## What changed and why

### 1. Subgroup assignment was wrong

Recovered from your Table 5's own arithmetic. It sums to 40,996 against a
40,430-pair dev set — an excess of 566, meaning the groups overlap:

```
|male ∩ female|             = 40,996 - 40,430           = 566
|male ∪ female|             = 1833 + 1751 - 566         = 3,018
union + name-only + neutral = 3,018 + 734 + 36,678      = 40,430   (= dev, exactly)
implied n_identity          = 3,018 + 734               = 3,752    (report says 3,751)
```

So in your paper: **male/female are decided by gendered *terms* — names don't
count** — a pair with both terms lands in **both** groups, and **name-only** means
a name with no gendered term.

My old code made `James` → male, which starved name-only to **14 examples**. Your
0.1324 gap was 11/14 correct on that group; one example moved it 7 points.

### 2. The pronoun mode is now switchable

Your flip rate came in at **0.0159** against the paper's **0.0296** — 46% lower.
One suspect: my `swap_identity` keeps counterfactuals grammatical
(`his credit score` → `her credit score`), while Table 1's literal mapping gives
`hers credit score`. Ungrammatical text is out-of-distribution for GPT-2 and may
flip predictions for reasons that have nothing to do with demographics.

This notebook runs **both** so you can measure it instead of speculating.

## Setup

In [ ]:
REPO = "https://github.com/Ricky11234/fairness-aware-gpt2"
CKPT_DATASET = "cda_reg"  # folder name of your checkpoint

import torch

assert torch.cuda.is_available(), "Set Accelerator to GPU T4 x2"
print(torch.cuda.get_device_name(0))

### Point this at your checkpoint

Easiest route: on your finished run's **Output**, click **New Dataset** to save
`checkpoints/` as a Kaggle Dataset, then **+ Add Input** it here. It mounts under
`/kaggle/input/<dataset-name>/`.

Run the cell below to find it.

In [ ]:
import subprocess

print(
    subprocess.run(
        ["find", "/kaggle/input", "-name", "head_config.json"], capture_output=True, text=True
    ).stdout
    or "No checkpoint found under /kaggle/input"
)

In [ ]:
# Paste the folder holding head_config.json (from the output above)
CKPT = "/kaggle/input/YOUR-DATASET-NAME/cda_reg"

from pathlib import Path

assert Path(CKPT, "head_config.json").exists(), f"No checkpoint at {CKPT}"
print("Checkpoint OK")

In [ ]:
%cd /kaggle/working
!rm -rf fairness-repo
!git clone -q $REPO fairness-repo
%cd fairness-repo
!pip install -q "transformers>=4.41" "datasets>=2.19"

In [ ]:
from pathlib import Path

src = Path("src/fairness_gpt2/identity.py").read_text()
assert "subgroups_of" in src, "Repo is stale — push the subgroup fix first"
print("Repo is current")

## Data

In [ ]:
!PYTHONPATH=src python scripts/download_data.py --only qqp --train-size 283011 --skip-test

## Run both evaluations

`contextual` keeps counterfactuals grammatical. `literal` reproduces Table 1's
mapping exactly. Same checkpoint, same dev set — the only difference is how
`his`/`her` are swapped.

In [ ]:
import os

os.environ["CKPT"] = CKPT
!PYTHONPATH=src python scripts/eval_checkpoint.py --ckpt $CKPT --dev data/quora-dev.csv --name cda_reg

In [ ]:
!PYTHONPATH=src python scripts/eval_checkpoint.py --ckpt $CKPT --dev data/quora-dev.csv --name cda_reg_literal --literal-pronouns

## Results

In [ ]:
import json
from pathlib import Path

ctx = json.loads(Path("results/reproduced/cda_reg.json").read_text())
lit = json.loads(Path("results/reproduced/cda_reg_literal.json").read_text())

PAPER = {
    "female": (1751, 0.8943),
    "male": (1833, 0.9127),
    "name-only": (734, 0.9373),
    "neutral": (36678, 0.8941),
}

print("SUBGROUP COUNTS (paper is 10-epoch; counts should still line up)")
print(f"{'group':<12}{'paper n':>9}{'yours':>9}{'diff':>8}")
for g in ("female", "male", "name-only", "neutral"):
    mine = ctx["per_subgroup"].get(g, {}).get("n", 0)
    print(f"{g:<12}{PAPER[g][0]:>9,}{mine:>9,}{mine - PAPER[g][0]:>+8,}")
tot = sum(v["n"] for v in ctx["per_subgroup"].values())
print(f"{'SUM':<12}{40996:>9,}{tot:>9,}   (over-sums by {tot - 40430:+,}; paper +566)")

print("\nFLIP RATE — grammatical vs the report's literal mapping")
print(f"  contextual (his->her) : {ctx['flip_rate']:.4f}  ({ctx['flips']} of {ctx['n_identity']})")
print(f"  literal    (his->hers): {lit['flip_rate']:.4f}  ({lit['flips']} of {lit['n_identity']})")
d = lit["flip_rate"] - ctx["flip_rate"]
print(f"  difference            : {d:+.4f}")
if lit["flips"]:
    print(
        f"  -> {d / lit['flip_rate']:.0%} of the literal flip rate is broken syntax, not identity"
    )
print("  paper (Table 4, 5 ep) : 0.0296")

print("\nHEADLINE")
print(f"{'metric':<15}{'paper':>10}{'yours':>10}{'delta':>10}")
for k, mine, p in (
    ("dev_acc", ctx["accuracy"], 0.8856),
    ("subgroup_gap", ctx["subgroup_gap"], 0.0512),
    ("flip_rate", ctx["flip_rate"], 0.0296),
):
    print(f"{k:<15}{p:>10.4f}{mine:>10.4f}{mine - p:>+10.4f}")

In [ ]:
import shutil

shutil.copytree("results/reproduced", "/kaggle/working/results/reproduced", dirs_exist_ok=True)
shutil.make_archive("/kaggle/working/reproduced", "zip", "results", "reproduced")
print("Download reproduced.zip from the Output panel")